Proyecto Final:
En este proyecto debes demostrar los conocimientos que has adquirido a lo largo de todo el programa. Deberás realizarlo una vez hayas completado todo el contenido obligatorio del programa y hayas entregado y superado todos los proyectos previos.

Objetivo del proyecto:
En este proyecto tienes que enfrentarte al folio en blanco. Es un proyecto libre donde tú marcas el objetivo de tus análisis. El proyecto se tiene que centrar en un EDA y Dashboard de un conjunto de datos a tu elección.

Procesos del proyecto:
En este proyecto deberás cubrir los siguientes puntos:
•	Transformación y limpieza profunda de los datos.
•	Análisis descriptivo de los datos.
•	Análisis estadístico de los datos.
•	Visualización de los datos.
•	Dashboard operativo.
•	Informe explicativo del análisis.

Herramientas para realizar el proyecto.
Tendrás que realizar el proyecto usando estás herramientas y librerías:
•	EDA y análisis de los datos:
o	Python
o	Pandas
o	Visual Studio Code
•	Dashboard y visualización de los datos. Puedes elegir hacerlo con alguna de estas herramientas:
o	Excel
o	Google Sheets
o	Power BI

Los datos:
Los datos con los que vas a trabajar son de carácter libre pero deben de cubrir todos los requisitos del proyecto de manera clara. Os dejamos algunos links para acceder a distintos conjuntos de datos que podéis usar en vuestro análisis:
•	https://www.kaggle.com/
•	https://datasetsearch.research.google.com/
•	https://registry.opendata.aws/
•	https://data.gov/
•	https://data.europa.eu/data/datasets?is_hvd=true&locale=en
•	https://www.ine.es/

Estos links son orientativos, puedes usar otras fuentes para extraer tus datos.

OBJETIVO DEL PROYECTO

En un negocio de gimnasio ser capaz de entender los datos históricos (analítica descriptiva) de:
-clientes: evolución temporal, caracterización, cuotas
-altas y bajas, caracterización
-visitas al gimnasio, ratios por día/horario, caracterización por tipo de cliente

Los datos está en un CRM ("Virtuagym"). Se emplearán las siguientes herramientas:
-carga de los datos:  Visual Studio Code, python con acceso API (requests, pandas), aplicando en módulo del programa "API"
-EDA: Visual Studio Code, python, pandas
-Dashboard y visualicación de datos: Power BI

PROYECTO

1. EDA: Compresión general de los datos

En este primer paso, el objetivo es familiarizarse con el conjunto de datos. Esto implica observar sus características generales, como el tamaño, el tipo de variables, la cantidad de valores nulos y la estructura básica de los datos. 
Las técnicas que se van a utilizar:
-Carga de los datos
-Tipos de datos
-Estadística descriptiva
-Resumen estadístico

1.1. EDA: Comprensión general de los datos. Carga de los datos.

ACCESO Y DESCARGA DE LOS DATOS

Los datos se encuentran un CRM, "Virtuagym".
Se solicita el acceso API al soporte de CRM, y este envía la documentación.
Es una API que requiere dos credenciales, "club_secret", y "api_key". Se obtienen.
En la documentación de la API, se identifican los siguientes Datasets de entre todos los disponibles, que tienen la información relevante:
Club Members:	Members of the club
Club Employees:	Employees of the club
Membership definition: Membership definition data
Membership Instances:	Membership instance data
Club Visits:	Checkins of Club Members

El acceso y descarga a los datos se realizará con python (con librería "requests" conforme al módulo del master), y la transformación, limpieza se realizará también con python (pandas - dataframe).

Proceso de acceso y descarga.

Como buena práctica de gestión de credenciales tal y como se indica en el curso, se almacena  en un archivo .env y se cargan utilizando  python-dotenv.
Se añade .env al .gitignore para no compartir este archivo en GIT.

Cada uno de los Datasets tiene sus endpoints, métodos, parámetros (como sistema de paginación) y cabeceras. Además, el API tiene una limitación por número máximo de consultas por hora, y por día. Por ello, para simplificar y optimizar el código,  se crean tres objetos:
-un objeto para gestionar el API, que vigila que no se ha excedido el número de solicitudes, y e implementa métodos de solicitud
-otro objeto para crear un dataframe a partir de un Dataset de la API

Se implementan además dos tipos de actualización de datos, un total "full" y otro incremental, según lo permitido por el API.
Todos los Datasets permiten actualización incremental salvo "Memberships Instances", que no tiene campo de timestamp. 
Para habilitar esta actualización full e incremental, para cada Dataset, se crean unos metadatos sobre si se ha implementado un backup full o incremental, el timestamp, y el cursor (en caso de incremental) que permite continuar un proceso de actualización ya iniciado. 

El almacenanamiento de las actualizaciones de cada Dataset se realiza en dos ficheros:
-uno general, que almacena los metadatos del Dataset en cabecera (#), y después una línea por cada dato, anadiéndose para cada dato si se recopiló por un backup full o incremental, así como el timestamp del api al actualizarse. Este general permite trazar todo el proceso de acceso y actualización en el API e incluso permitiría trazar variaciones históricas en algunos datos
-un snapshot, que es sólo la versión de los datos más reciente, para después ser analizado y transformado

El proceso de acceso y descarga termina generado 5 dataframes, que son los snapshots de los Datasets:
df_members (Club Members)
df_employees (Club Employees)
df_memberships_definition (Membership definition)
df_memberships (Membership Instances)
df_visits (Club Visits)

Se da por concluída esta fase de acceso y descarga de los datos.

(VER CÓDIGO PYTHON ANEXO)


In [1]:
# se cargan las credenciales de acceso de la API que se almacenan en un fichero .env que se añadido al .gitignore
from dotenv import load_dotenv
import os
import datetime as dt


# Cargar la claves API desde el archivo .env
load_dotenv()
club_id = os.getenv("CLUB_ID")
club_key = os.getenv("CLUB_KEY")
vg_api_key = os.getenv("VG_API_KEY")

In [2]:
#se realiza una función para crear urls dado que hay distintos endpoints para cada tabla,
# y existe paginación 

def crear_url(endpoint, paginacion_tipo, pagina_valor=0):
    endpoints={'Members':'member', 'Membership Instances':'membership/instance', 'Membership Definition':'membership/definition', 'Club Visits':'visits', 'Club Employees':'employee', "Invoices":'invoices'}
    url=f"https://api.virtuagym.com/api/v1/club/{club_id}/{endpoints.get(endpoint)}?api_key={vg_api_key}&club_secret={club_key}&{paginacion_tipo}={pagina_valor}"
    print(url)
    return url


In [3]:
import time
import requests

class GestorAPI:
    def __init__(self, archivo_log="api_log.txt", límite_hora=500, límite_día=1000):
        self.log = archivo_log
        self.límite_hora = límite_hora
        self.límite_día = límite_día
        self.historial= self.cargar_historial(self.log)
    def ahora_ms(self):
        return int(time.time()*1000)
    def purgar_historial(self):
        self.historial = [int(t) for t in self.historial if self.ahora_ms()-t <= 24*60*60*1000]
        with open(self.log, "w") as f:
            f.write(",".join(map(str, self.historial)))
    def cargar_historial(self, log):
        self.historial =[]
        if os.path.exists(log):
            with open(log, "r") as f:
                raw = [x for x in f.read().strip().split(",") if x.strip()]
                if raw:
                    self.historial = [int(x) for x in raw]
                self.purgar_historial()
        return self.historial
    def api_status(self):
            ahora_ms=self.ahora_ms()
            solicitudes_última_hora= len([t for t in self.historial if ahora_ms-t <= 60*60*1000])
            solicitudes_último_día = len([t for t in self.historial if ahora_ms- t < 24*60*60*1000])
            solicitudes_hora_disponibles= self.límite_hora- solicitudes_última_hora
            solicitudes_día_disponibles= self.límite_día - solicitudes_último_día
            return (solicitudes_última_hora, solicitudes_último_día, solicitudes_hora_disponibles, solicitudes_día_disponibles)
    def api_request(self, url):
        self.purgar_historial()
        _,_,disp_hora, disp_día=self.api_status()
        if  (disp_hora>0) and (disp_día>0):
            ahora_ms=self.ahora_ms()
            self.historial.append(ahora_ms)
            with open(self.log, "w") as f:
                f.write(",".join(map(str, self.historial)))
            try:
                response = requests.get(url)
                response.raise_for_status()
                return (1, response.json())
            except requests.exceptions.RequestException as e: 
                 return (-1, e)
        else:
            return (0, disp_hora, disp_día)

In [4]:
DATASETS = {
    "Members": {
        "index": "member_id",
        "timestamp_field": "timestamp_edit",             
    },
    "Membership Definition": {
        "index": "membership_id",
        "timestamp_field": "membership_last_edit_date",   
    },
    "Membership Instances": {
        "index": "instance_id",
        "timestamp_field": None,                         
    },
    "Club Visits": {
        "index": "id",
        "timestamp_field": "check_in_timestamp",                      
    },
    "Club Employees": {
        "index": "member_id",
        "timestamp_field": "timestamp_edit",
    },
}

In [5]:
import pandas as pd

class API_Dataset:
    def __init__(self, tipo, api, archivo):
        self.tipo=tipo
        self.index=DATASETS.get(tipo)['index']
        self.timestamp_field=DATASETS.get(tipo)['timestamp_field']
        self.timestamp=0
        self.api=api
        self.archivo=archivo
        self.full_completo=False
        self.incremental_completo=False
        self.sync_mode='full'
        self.sync_completo=True
        self.paginacion_tipo=''
        self.cursor=0
        self.datos=self.cargar_dataset()

    def _attrs_header(self):
        return [ k for k, v in self.__dict__.items() if k not in {"datos", "api"} and v not in (None, "")]

    def cargar_dataset(self):
        if not(os.path.exists(self.archivo)):
            return pd.DataFrame()
        with open(self.archivo, "r", encoding="utf-8") as f:
            header = f.readline().strip()
            if header.startswith("#"):
                for pair in header[1:].split(","):
                    k, v = pair.split("=", 1)
                    if not hasattr(self, k):
                        continue
                    tipo_atributo = type(getattr(self, k.strip()))
                    if tipo_atributo==bool:
                        v= v.strip().lower()=="true"
                    else:
                        v=tipo_atributo(v.strip())
                    setattr(self, k.strip(), v)
            df = pd.read_csv(f)
            df.set_index(self.index, inplace=True, drop=False)
        return df
        
    def guardar_dataset(self):
        attrs=self._attrs_header()
        with open(self.archivo, "w", encoding="utf-8", newline="") as f:
          f.write("# " + ",".join(f"{a}={getattr(self, a)}" for a in attrs) + "\n")
          self.datos.to_csv(f, index=False)
        if self.sync_completo:
            self.hacer_snapshot()

    def actualizar_status(self):
        log='dataset_status.csv'
        dataset_status={}
        if os.path.exists(log):
            with open(log, "r", encoding="utf-8") as f:
                datos = f.readline().strip()
                if datos:
                    for pair in datos.split(","):
                        dataset, timestamp = pair.split("=", 1)
                        dataset_status[dataset]=int(timestamp)
        dataset_status[self.tipo]=int(self.timestamp)
        with open(log, "w", encoding="utf-8", newline="") as f:
                f.write(",".join(f"{ds}={ts}" for ds, ts in dataset_status.items()))
    
    def dataset_update(self, sync_mode):
        if (sync_mode=='incremental') and (self.full_completo==False):
            return (0, "imposible incremental porque todavía no hay un full")
        if (sync_mode=='incremental'):
            if  (self.timestamp_field==None):
                return (0, "este dataset no soporta backup incremental")
            else:
                self.cursor=max(self.cursor, self.timestamp)
        self.sync_mode=sync_mode
        if self.paginacion_tipo=='':
            if sync_mode=='full':
                self.paginacion_tipo='from_id'
            else:
                self.paginacion_tipo='sync_from'
        if self.sync_completo:
            self.sync_completo=False
        while not self.sync_completo:
            url=crear_url(self.tipo, self.paginacion_tipo, self.cursor)
            respuesta_api=self.api.api_request(url)
            if respuesta_api[0]==1:
                datos=pd.DataFrame(respuesta_api[1]['result'])
                if datos.empty:
                    self.sync_completo=True
                    self.actualizar_status()
                    self.cursor=0
                    return (1, "update realizado pero sin actualización de datos")
                datos['api_timestamp']=int(respuesta_api[1]['status']['timestamp'])
                datos['sync_mode']=sync_mode
                datos.set_index(self.index, inplace=True, drop=False)
                self.datos=pd.concat([self.datos, datos], axis=0)
                try:
                    results_remaining= int(respuesta_api[1]['status']['results_remaining'])
                    print(f"results remaining: {results_remaining}")
                except:
                    results_remaining=0
                self.sync_completo = (results_remaining==0)
                if self.sync_completo:
                    self.cursor=0
                    self.paginacion_tipo=''
                    print("se ha completado la actualización")
                    if sync_mode=='full':
                        self.full_completo=True
                    else:
                        self.incremental_completo=True
                    if self.timestamp_field==None:
                        self.timestamp=0
                    else: 
                        col = self.datos[self.timestamp_field]
                        if not pd.api.types.is_numeric_dtype(col):
                            col = pd.to_datetime(col, errors="coerce").astype("int64") // 10**9
                        self.timestamp = int(col.max())
                    self.actualizar_status()
                    self.guardar_dataset()
                    return (1, "update realizado completamente")
                else:
                    next_page=respuesta_api[1]['status']['next_page']
                    print(next_page)
                    if type(next_page)==int:
                        self.cursor=next_page
                        self.paginacion_tipo='page'
                    else:
                        self.cursor=int(respuesta_api[1]['status']['next_page'].split('=')[1])
                        self.paginacion_tipo=str(respuesta_api[1]['status']['next_page'].split('=')[0])
                    print(f"seguimos interando cursor: {self.cursor}")
                self.guardar_dataset()
            else:
                return (0, "el api no respondió, verficar status del api")

    def hacer_snapshot(self):
        if (self.sync_completo==True) and not(self.datos.empty):
            old = self.datos.index.name
            self.datos.index.name = "_row"
            snapshot = (self.datos.sort_values("api_timestamp").groupby(self.index, as_index=False, sort=False).tail(1))
            snapshot = snapshot.drop(columns=["sync_mode", "api_timestamp"], errors="ignore")
            with open(self.archivo[:len(self.archivo)-len('.csv')]+'_snapshot.csv', "w", encoding="utf-8", newline="") as f:
                attrs=self._attrs_header()
                f.write("# " + ",".join(f"{a}={getattr(self, a)}" for a in attrs) + "\n")
                snapshot.to_csv(f, index=False)
            self.datos.index.name = old
            return (1, snapshot)
        return 0
    
    def cargar_snapshot(self):
        file=self.archivo[:len(self.archivo)-len('.csv')]+'_snapshot.csv'
        if not(os.path.exists(file)):
            return (0, {}, pd.DataFrame())
        meta={}
        with open(file, "r", encoding="utf-8") as f:
            header = f.readline().strip()
            if header.startswith("#"):
                for pair in header[1:].split(","):
                    k, v = pair.split("=", 1)
                    if not hasattr(self, k):
                        continue
                    tipo_atributo = type(getattr(self, k.strip()))
                    if tipo_atributo==bool:
                        v= v.strip().lower()=="true"
                    else:
                        v=tipo_atributo(v.strip())
                    meta[k.strip()]=v
            df = pd.read_csv(f)
            df.set_index(self.index, inplace=True, drop=False)
            return (1, meta, df)

    

In [6]:
api_vg= GestorAPI()

In [7]:
Employees= API_Dataset('Club Employees', api_vg, 'Club Employees.csv')


In [8]:
Employees.dataset_update('incremental')

(0, 'imposible incremental porque todavía no hay un full')

In [9]:
print (api_vg.api_status())

(0, 12, 500, 988)


In [10]:
Members= API_Dataset('Members', api_vg, 'Members.csv')

In [11]:
Members.dataset_update('incremental')

(0, 'imposible incremental porque todavía no hay un full')

In [13]:
Memberships_definition= API_Dataset('Membership Definition', api_vg, 'Membership Definition.csv')


In [14]:
Memberships_definition.dataset_update('incremental')

(0, 'imposible incremental porque todavía no hay un full')

In [15]:
Memberships= API_Dataset('Membership Instances', api_vg, 'Membership Instances.csv')

In [16]:
Memberships.dataset_update('full')

https://api.virtuagym.com/api/v1/club/79788/membership/instance?api_key=84wLPfdxbmKmoh9amodLKpcJD2RIikwJWuu5KjkE4o&club_secret=CS-79788-ACCESS-TJnuiYO7hB1sNxhDutBZsdRye&from_id=0
results remaining: 2850
from_id=9489404
seguimos interando cursor: 9489404
https://api.virtuagym.com/api/v1/club/79788/membership/instance?api_key=84wLPfdxbmKmoh9amodLKpcJD2RIikwJWuu5KjkE4o&club_secret=CS-79788-ACCESS-TJnuiYO7hB1sNxhDutBZsdRye&from_id=9489404
results remaining: 2350
from_id=9507466
seguimos interando cursor: 9507466
https://api.virtuagym.com/api/v1/club/79788/membership/instance?api_key=84wLPfdxbmKmoh9amodLKpcJD2RIikwJWuu5KjkE4o&club_secret=CS-79788-ACCESS-TJnuiYO7hB1sNxhDutBZsdRye&from_id=9507466
results remaining: 1850
from_id=10179376
seguimos interando cursor: 10179376
https://api.virtuagym.com/api/v1/club/79788/membership/instance?api_key=84wLPfdxbmKmoh9amodLKpcJD2RIikwJWuu5KjkE4o&club_secret=CS-79788-ACCESS-TJnuiYO7hB1sNxhDutBZsdRye&from_id=10179376
results remaining: 1350
from_id=115702

(1, 'update realizado completamente')

In [17]:
Visits= API_Dataset('Club Visits', api_vg, 'Club Visits.csv')

In [18]:
Visits.dataset_update('incremental')

(0, 'imposible incremental porque todavía no hay un full')

In [21]:
#vía API Web se hace un snapshot (versión más reciente) de los datasets, y se crea un dataframe para cada uno 
# (1) dataframe df_members, miembros: "Members"
# (2) dataframe df_employees, empleados: "Club Employees", 
# (3) dataframe df_memerbships_definition, definición de cada tipo de membresía "Membership Defintion"
# (4) dataframe df_memberships, membresía asociada a cada miembro: "Membership Instances"
# (5) dataframe df_visits, visitas al centro: "Club Visits",

resultado, meta, members=Members.cargar_snapshot()
if resultado==1:
    df_members=members
else:
    print("error al cargar snapshot Members")

resultado, meta, employees=Employees.cargar_snapshot()
if resultado==1:
    df_employees=employees
else:
    print("error al cargar snapshot Employees")

resultado, meta, memberships_definition=Memberships_definition.cargar_snapshot()
if resultado==1:
    df_memberships_definition=memberships_definition
else:
    print("error al cargar snapshot Memberships_definition")

resultado, meta, memberships=Memberships.cargar_snapshot()
if resultado==1:
    df_memberships=memberships
else:
    print("error al cargar snapshot Memberships")

resultado, meta, visits=Visits.cargar_snapshot()
if resultado==1:
    df_visits=visits
else:
    print("error al cargar snapshot Visits")


1.2. EDA: Comprensión general de los datos. Vista rápida. Tipos de datos. Estadística descriptiva. Resumen estadístico.

Esta fase se realiza con los siguientes métodos de pandas:
df.head()       # primeras filas
df.info()       # tipos de datos, nulos
df.describe()   # estadísticos numéricos básicos


Procedemos con los todos los datasets, 

CLUB MEMBERS

Se compone de los siguientes campos, según el API,
Field	Type	Optional	Values	Information
1. member_id	int	no		The ID for the member ("Member ID" in Virtuagym)
2. club_id	int	no		The ID of the club the user belongs to in Virtuagym (in case of a chain this is a sub-club)
3. club_member_id	string	yes		Custom ID from the external system ("Own member ID" in Virtuagym)
4. external_id	string	yes		ID for the member from the external system
5. firstname	string	no		The first name of the member
6. lastname	string	no		Last name of the member
7. email	string	no		The email address of the member, an invitation will be sent to this address when the member will be created (If enabled in System Settings→ Coach → Module Settings → Auto-invite newly added clients)
8. active	boolean	no		If the member is active or inactive
9. is_pro	boolean	no		If the member is pro
10. gender	string	yes	"m"/"f"	The gender of the editted member
11. member_since	int	no		The timestamp (in milliseconds) the member made an account (yyyy-mm-dd)
12. timestamp_edit	int	no		The timestamp (in milliseconds) the members' information changed. The timestamp_edit has to be newer than the value we have saved in the database, otherwise we will not update (but still return success). We use it to make sure the app does not sync back old data and overwrite newer change made in the web platform.The best option for external services and for anyone not trying to keep two copies of this data is to just leave it off entirely (or empty, if they can’t for some reason).
13. birthday	string	yes		The birthday of the member (YYYY-MM-DD)
14. lang	string	yes		The language the member uses in the portal
15. zip	string	yes		The zip code of the member
16. street	string	yes		The street address of the member
17. street_extra	string	yes		Extra street information of the member
18. place	string	yes		The place where the member lives
19. country	string	yes		The country code where the member lives
20. formatted_address	string	yes		A nicely formatted string of the street address of the member
21. phone	string	yes		The phone number of the member
22. mobile	string	yes		The mobile phone number of the member
23. rfid_tag	string	yes		Rf-ID tag that is tied to the member
24. early_booking_access	boolean	yes		If the member has early booking access

Se realiza un vista general con df.head(), df.info(), df.describe().

Las conclusiones son:
Hay varios campos que no aportan ninguna información porque tienen el mismo valor (country, lang, early_access)
Las columnas son iguales a las de la API salvo que "external_id" es igual a "orginal_member_id", y los tipos de datos en la API "string" en el dataframe es "object". 
Será necesario en fases posteriores : (0) fijar el índice al "member_id" que es el id único de los clientes (1) eliminar campos por protección de datos (firstname, lastname, email, phone, mobile), (2) eliminar campos no relevantes (club_id, external_id, is_pro, lang, country, rfid_tag, early_booking_access, user_id, formatted_address), y (3) convertir datos a string (son object) y a fechas (son números o string)



CLUB EMPLOYEES

Los empleados son miembros. El dataset "Club Employees" identifica a los empleados del centro, para poderlo eliminarlo del análisis ya que lo puede distorsionar puesto que no son clientes.

Se compone de los siguientes campos, según el API,
Field	Type	Optional	Values	Information
1. member_id	int	no		The ID for the employee("Member ID" in Virtuagym)
2. club_id	int	no		The ID of the club the user belongs to in Virtuagym (in case of a chain this is a sub-club)
3. club_member_id	string	yes		Custom ID from the external system ("Own member ID" in Virtuagym)
4. external_id	string	yes		ID from the external system
5. firstname	string	no		The first name of the employee
6. lastname	string	no		Last name of the employee
7. email	string	no		The email address of the employee, an invitation will be sent to this address when the employee will be created
8. active	boolean	no		If the employeeis active or inactive
9. is_pro	boolean	no		If the employeeis pro
10. gender	string	yes	"m"/"f"	The gender of the editted employee
11. member_since	int	no		The timestamp (in milliseconds) the employeemade an account (yyyy-mm-dd)
12. timestamp_edit	int	no		The timestamp (in milliseconds) the employee' information changed
13. birthday	string	yes		The birthday of the employee(YYYY-MM-DD)
14. lang	string	yes		The language the employee uses in the portal
15. zip	string	yes		The zip code of the employee
16. street	string	yes		The street address of the employee
17. street_extra	string	yes		Extra street information of the employee
18. place	string	yes		The place where the employee lives
19. country	string	yes		The country code where the employee lives
20. formatted_address	string	yes		A nicely formatted string of the street address of the employee
21. phone	string	yes		The phone number of the employee
22. mobile	string	yes		The mobile phone number of the employee
23. rfid_tag	string	yes		Rf-ID tag that is tied to the employee

Se realiza un vista general con df.info(). Esta tabla es sólo para identificar a los empleados y eliminarlos del análisis de miembros.
En efecto a través del campo member_id, se pueden identificar los empleados.

Las conclusiones son:
Se verifican los empleados, personal del gimnasio y también personal de mantemiento.


MEMBERSHIP DEFINITION

Este dataset es importante porque contiene la definición de membresía; de aquí se puede extraer la precio de cada membresía y su nombre.

Se compone de los siguientes campos, según el API,

Field	Type	Values	Description
1. membership_id	integer	-	Unique identifier for the membership.
2. membership_name	string	-	Name of the membership.
3. membership_group	string	-	Group the membership belongs to.
4. membership_notes	string	-	Additional notes for the membership.
5. membership_availability_start	date	-	Start date of membership availability.
6. membership_availability_end	date	-	End date of membership availability.
7. membership_available_online	boolean	true, false	Whether the membership is available online.
8. membership_duration	integer	-	Duration of the membership.
9. membership_duration_type	string	weeks, months	Type of duration.
10. membership_auto_renew	boolean	true, false	Whether the membership auto-renews.
11. membership_pro_rata_start	boolean	true, false	Whether pro-rata billing is applied at the start.
12. membership_renew_duration	integer	-	Duration for renewal.
13. membership_renew_term	string	weeks, months	Term for renewal duration.
14. membership_renew_before	integer	-	Days before renewal.
15. membership_renew_before_term	string	days, weeks, months	Term for renewal before.
16. membership_renew_price	decimal	-	Price for renewal.
17. membership_price	decimal	-	Price of the membership.
18. membership_price_term	string	total, monthly, weekly, four_weekly	Term for the price.
19. membership_income_category	string	-	Income category for the membership.
20. membership_registration_fee	decimal	-	Registration fee for the membership.
21. membership_club_tax	object	-	Tax information for the membership. (see below)
22. membership_billing_cycle	string	-	Billing cycle for the membership.
22. default_payment_method	string	-	Default payment method for the membership.
23. tmp_default_payment_method	string	-	Temporary default payment method for the membership.
24. membership_creation_date	date	-	Date the membership was created.
25. membership_last_edit_date	date	-	Date the membership was last edited.
26. membership_invoice_creation_term	string	-	Term for invoice creation.
27. access_times	array	-	Access times for the membership. (see below)

Se realiza un vista general con df.head(), df.info(), df.describe().

Las conclusiones son: (1) se identifican los distintos tipos de membresía (2) hay que cambiar el tipo de membership_name (tipo object) a string (3) hay que eliminar todos los campos salvo membership_id, membership_name, membership_price, membership_registration_fee


MEMBERSHIP INSTANCES

Este dataset es importante porque contiene la asociación de miembro a tipo de membresía, la fecha de alta, de baja si existe por cada tipo, y de ahí se puede determinar también si se está activo o no.

Se compone de los siguientes campos, según el API,
Field	Type	Optional	Values	Information
1. instance_id	int	no		The id of the membership instance
2. member_id	int	no		The id of the [[Club Members]]
3. membership_id	int	no		The id of the [[Membership Definition]]
4. active	boolean	no		If the membership instance is active or inactive
5. cancelled	boolean	no		If the membership was manually cancelled by an employee with termination in the future
6. contract_autorenewed	boolean	no		If the membership was already renewed at this point or it's still in its initial period
7. completed	boolean	no		If the membership has reached the contract end date automatically and was not renewed
8. paused	boolean	no		If the membership instance was paused by an employee
9. stopped	boolean	no		If the membership was manually cancelled by an employee with immediate termination
10. start_date	string	no		The actual start date of the membership (yyyy-mm-dd)
11. contract_start_date	string	no		The contractual start date of the membership (yyyy-mm-dd)
12. contract_end_date	string	no		The contractual end date of the membership (yyyy-mm-dd)
13. membership_name	string	no		The name of the membership

Se realiza un vista general con df.head(), df.info(), df.describe().

Las conclusiones son: (1) se identifican los miembros con membresía asociada con su membresía (nombres únicos), (2) hay que cambiar los tipos de datos de las fechas (tipo object) a datetime (3) hay que quitar muchos campos que no se emplearán como membership_id, active, cancelled, contract_autorenewed, completed, paused, stopped, start_time


CLUB VISITIS

Este dataset es importante porque contiene la asistencia de los clientes al gimnasio. 
Sería posible entender patrones de visitas de horas-días, frecuencias, duraciones, diferencias en asistenca por sexo -edad, patrones de baja asociados a la frecuencia de visita, horarios, etc.
Además, otra cuestión es detectar posibles accesos no permitidos (pej múltiples check-in sin check-out, acceso con diferentes dispositivos)

Se compone de los siguientes campos, según el API,
Field	Type	Values	Information
1. id	int		The unique identifier of the visit
2. club_id	int		The id of the corresponding club
3. member_id	int		The id of the member who triggered the visit
4. check_in_timestamp	int		The timestamp (in ms) of the check-in
5. check_out_timestamp	int		The timestamp (in ms) of the check-out
6. status	string	ok, warning, rejected	A human readable status of the visit
7. status_message	string		Supplementary text of the visit


Se realiza un vista general con df.head(), df.info(), df.describe().

Como conclusiones son: 1/ hay un campo adicional, device_id, dispositivo con el que se accede que podría explotarse para identifacar accesos indebidos (se facilita el usuario de la aplicación a otra persona por ejemplo), 2/ se pueden eliminar los campos status, status_id es la membresía que es un dato que no aporta, se puede eliminar, las fechas están como número entero, hay que convertirlas a datetime.

(VER CÓDIDO PYTHON ANEXO)



In [22]:
#se habilita la opción de que se muestren todas las columnas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

In [23]:
df_members.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4146 entries, 45986922 to 45988359
Data columns (total 27 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   member_id             4146 non-null   int64  
 1   firstname             4146 non-null   object 
 2   lastname              4146 non-null   object 
 3   active                4146 non-null   bool   
 4   is_pro                4146 non-null   bool   
 5   gender                4146 non-null   object 
 6   email                 4142 non-null   object 
 7   member_since          4146 non-null   int64  
 8   timestamp_edit        4146 non-null   int64  
 9   country               4146 non-null   object 
 10  rfid_tag              10 non-null     object 
 11  club_id               4146 non-null   int64  
 12  registration_date     4146 non-null   int64  
 13  lang                  4146 non-null   object 
 14  original_member_id    4146 non-null   int64  
 15  early_booking_a

In [25]:
df_members.head()

,member_id,firstname,lastname,active,is_pro,gender,email,member_since,timestamp_edit,country,rfid_tag,club_id,registration_date,lang,original_member_id,early_booking_access,user_id,vat_number,birthday,zip,street,place,mobile,street_extra,formatted_address,club_member_id,phone
member_id,,,,,,,,,,,,,,,,,,,,,,,,,,,
45986915,45986915,Julio,Carrero Mellado,False,False,u,julione_somoto@hotmail.com,1535587200000,1712582761338,ES,NaN,79788,1711580175,es,0,False,NaN,NaN,1991-12-09,28912,Calle San Clemente,Leganés,601166222,NaN,NaN,53716861,NaN
45986914,45986914,Moisés,Ionut Flaviu,False,False,u,flaviu@gefirenet.com,1528329600000,1712582761327,ES,NaN,79788,1711580175,es,0,False,NaN,NaN,1997-08-14,28914,Calle Alcalde Pedro González González 15,Leganés,603048221,NaN,NaN,8389679,NaN
45986913,45986913,Silvia Mercedes,Martín Gordo,False,False,u,silvia.martin@vialos.es,1572307200000,1712582761316,ES,NaN,79788,1711580175,es,0,False,NaN,NaN,1969-08-20,28914,Calle Islas Canarias,Leganés,661224189,NaN,NaN,52121767,NaN
45986912,45986912,Jennifer,Rubio Dochado,False,False,f,jeeennifer10@gmail.com,1620432000000,1757923124200,ES,NaN,79788,1711580175,es,0,False,NaN,NaN,1994-11-12,28914,Calle Federica Montseny,Leganés,685733321,"4, 3B",NaN,53721812,NaN
45824256,45824256,MICKEY MOUSE,MOUSE,False,False,m,hola@hola.com,1710460800000,1717241880022,ES,NaN,79788,1710528875,es,0,False,NaN,NaN,1991-01-01,NaN,NaN,NaN,644367953,NaN,NaN,NaN,NaN


In [26]:
df_members.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
member_id,4136.0,NaN,NaN,NaN,47557095.296663,3042961.215135,44229454.0,45987935.75,45988969.5,47135103.75,57617144.0
firstname,4136,1127,David,94,NaN,NaN,NaN,NaN,NaN,NaN,NaN
lastname,4136,3544,López García,8,NaN,NaN,NaN,NaN,NaN,NaN,NaN
active,4136,2,False,2839,NaN,NaN,NaN,NaN,NaN,NaN,NaN
is_pro,4136,2,False,2849,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,4136,4,u,1884,NaN,NaN,NaN,NaN,NaN,NaN,NaN
email,4132,4099,larabeatriz.silva@outlook.com,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
member_since,4136.0,NaN,NaN,NaN,1628671961315.280518,189140700965.345551,0.0,1564876800000.0,1651622400000.0,1718928000000.0,1778544000000.0
timestamp_edit,4136.0,NaN,NaN,NaN,1739913117885.60083,26218993200.996357,1705338988561.0,1712582780087.75,1739415224506.5,1766347596545.5,1778601357296.0
country,4136,1,ES,4136,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [27]:
df_employees.info()

<class 'pandas.core.frame.DataFrame'>
Index: 20 entries, 44229454 to 57501590
Data columns (total 27 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   member_id             20 non-null     int64  
 1   firstname             20 non-null     object 
 2   lastname              20 non-null     object 
 3   active                20 non-null     bool   
 4   is_pro                20 non-null     bool   
 5   gender                20 non-null     object 
 6   email                 20 non-null     object 
 7   member_since          20 non-null     int64  
 8   timestamp_edit        20 non-null     int64  
 9   country               20 non-null     object 
 10  rfid_tag              10 non-null     object 
 11  club_id               20 non-null     int64  
 12  registration_date     20 non-null     int64  
 13  lang                  20 non-null     object 
 14  original_member_id    20 non-null     int64  
 15  early_booking_acc

In [20]:
df_employees.head(20)

,member_id,firstname,lastname,active,is_pro,gender,email,member_since,timestamp_edit,country,rfid_tag,club_id,registration_date,lang,original_member_id,early_booking_access,user_id,vat_number,priviliges,birthday,zip,street,place,mobile,street_extra,formatted_address,club_member_id
member_id,,,,,,,,,,,,,,,,,,,,,,,,,,,
44229454,44229454,Antonio,Fernández,True,True,u,afborondo@hotmail.com,1700697600000,1759497786772,ES,48-91-A1-99-90-00-00-00-00-00-00-00-00-00-00-00,79788,1700755326,es,0,False,26979635.0,NaN,"default,club_manager,scheduling,coach",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
44229588,44229588,LUIS,BADOS,True,True,m,luisbados@gmail.com,1700697600000,1721228805973,ES,48-7F-FD-99-90-00-00-00-00-00-00-00-00-00-00-00,79788,1700755721,es,0,False,27466928.0,NaN,"default,financial,club_manager,assistent_manager,employee,credit_manager,sales,scheduling,marketing_manager,coach,community_manager",1977-01-18,NaN,NaN,NaN,NaN,NaN,NaN,NaN
44229613,44229613,BELEN,PARDO,True,True,f,belenpardoramirez@hotmail.com,1700697600000,1747935542833,ES,vg_checkin_qr=mwqnB020JTqkPEoyT6qNon5TPVGdGo+Iw,79788,1700755822,es,0,False,32186512.0,NaN,"default,financial,club_manager,assistent_manager,employee,credit_manager,sales,scheduling,marketing_manager,coach,community_manager",1985-12-03,28970.0,"Calle Azulejo 35, 3B",Humanes de Madrid,6.573642e+08,NaN,NaN,NaN
44229651,44229651,LARA,SILVA,False,False,f,larabeatriz.silva@outlook.com,1700697600000,1705338988561,ES,NaN,79788,1700755979,es,0,False,NaN,NaN,"default,financial,club_manager,assistent_manager,employee,credit_manager,sales,scheduling,marketing_manager,coach,community_manager",1994-08-27,NaN,NaN,NaN,NaN,NaN,NaN,NaN
44842160,44842160,LARA,SILVA,True,True,f,larabeatriz.silva@outlook.com,1705276800000,1724691942094,ES,28-E5-35-99-90-00-00-00-00-00-00-00-00-00-00-00,79788,1705339062,es,0,False,27468263.0,NaN,"default,financial,club_manager,assistent_manager,employee,credit_manager,sales,scheduling,marketing_manager,coach,community_manager",1994-08-27,NaN,NaN,NaN,6.443680e+08,NaN,NaN,NaN
44854064,44854064,FABIO AUGUSTO,SILVA,True,True,m,fabio_fas@live.com,1705363200000,1724692001997,ES,B8-E8-87-98-90-00-00-00-00-00-00-00-00-00-00-00,79788,1705397460,es,0,False,27479111.0,NaN,"default,financial,club_manager,assistent_manager,employee,credit_manager,sales,scheduling,marketing_manager,coach,community_manager",1992-02-09,NaN,NaN,NaN,NaN,NaN,NaN,NaN
44961752,44961752,ADRIAN,FERNANDEZ,True,True,m,adriansilvabr01@gmail.com,1705881600000,1744250815274,ES,NaN,79788,1705949777,es,0,False,32145407.0,NaN,"default,financial,club_manager,assistent_manager,employee,credit_manager,sales,scheduling,marketing_manager,coach,community_manager",2002-06-16,NaN,NaN,NaN,6.011204e+08,NaN,NaN,NaN
46914433,46914433,Lupe,Flores Luengo,True,True,f,lupetala@hotmail.com,1717718400000,1727682500707,ES,NaN,79788,1717780551,es,0,False,8296705.0,NaN,"default,coach",1972-05-09,28021.0,calle oxigeno 5,madrid,6.496726e+08,NaN,NaN,NaN
47103363,47103363,Luciana,Benedetti Silva,True,True,f,lsilvabenedetti@gmail.com,1719187200000,1765472169834,ES,NaN,79788,1719251541,es,0,False,29222080.0,NaN,default,1970-08-11,28914.0,Calle Alcalde Alfredo de Castro 28,Leganés,6.443578e+08,"Portal 1, 3D","C. del Alcalde Alfredo de Castro, 28, 28914 Leganés, Madrid, Spain",NaN


In [69]:
df_memberships_definition.info()

<class 'pandas.core.frame.DataFrame'>
Index: 36 entries, 285058 to 292086
Data columns (total 27 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   membership_id                     36 non-null     int64  
 1   membership_name                   36 non-null     object 
 2   membership_group                  36 non-null     object 
 3   membership_availability_start     21 non-null     object 
 4   membership_availability_end       21 non-null     object 
 5   membership_available_online       36 non-null     bool   
 6   membership_duration               36 non-null     int64  
 7   membership_duration_type          36 non-null     object 
 8   membership_auto_renew             36 non-null     bool   
 9   membership_pro_rata_start         36 non-null     bool   
 10  membership_renew_duration         33 non-null     float64
 11  membership_renew_term             33 non-null     object 
 12  member

In [21]:
df_memberships_definition.head(5)

,membership_id,membership_name,membership_group,membership_availability_start,membership_availability_end,membership_available_online,membership_duration,membership_duration_type,membership_auto_renew,membership_pro_rata_start,membership_renew_duration,membership_renew_term,membership_renew_before,membership_renew_before_term,membership_renew_price,membership_price,membership_price_term,membership_income_category,membership_registration_fee,membership_club_tax,membership_billing_cycle,default_payment_method,tmp_default_payment_method,membership_creation_date,membership_last_edit_date,membership_invoice_creation_term,access_times
membership_id,,,,,,,,,,,,,,,,,,,,,,,,,,,
289796,289796,CUOTA PRO CUARTO FAMILIAR,default,1970-01-01,1970-01-02,False,1,months,True,False,1.0,months,7.0,days,32,32,monthly,memberships,20.0,"{'tax_id': '1', 'tax_name': 'IVA 21%', 'tax_percentage': 21}",monthly,directdebit_NL,direct_debit,2024-03-25,2025-02-26,7 days,NaN
285068,285068,CUOTA ARTES MARCIALES + GYM,default,1970-01-01,1970-01-02,False,1,months,True,True,1.0,months,7.0,days,82,82,monthly,memberships,20.0,"{'tax_id': '1', 'tax_name': 'IVA 21%', 'tax_percentage': 21}",monthly,directdebit_NL,direct_debit,2024-02-06,2025-02-26,1 weeks,NaN
289800,289800,CUOTA CUARTO FAMILIAR FUNDADOR,default,NaN,NaN,False,1,months,True,False,1.0,months,6.0,days,23,23,monthly,memberships,20.0,"{'tax_id': '1', 'tax_name': 'IVA 21%', 'tax_percentage': 21}",monthly,directdebit_NL,direct_debit,2024-03-25,2026-02-22,4 days,NaN
285038,285038,CUOTA PRO TERCER FAMILIAR,default,1970-01-01,1970-01-02,False,1,months,True,True,1.0,months,7.0,days,42,42,monthly,memberships,20.0,"{'tax_id': '1', 'tax_name': 'IVA 21%', 'tax_percentage': 21}",monthly,directdebit_NL,direct_debit,2024-02-06,2025-02-26,1 weeks,NaN
285036,285036,CUOTA PRO MENSUAL,default,1970-01-01,1970-01-02,False,1,months,True,True,1.0,months,7.0,days,55,55,monthly,memberships,20.0,"{'tax_id': '1', 'tax_name': 'IVA 21%', 'tax_percentage': 21}",monthly,directdebit_NL,direct_debit,2024-02-06,2025-02-26,1 weeks,NaN


In [22]:
df_memberships_definition.describe(include='all')

,membership_id,membership_name,membership_group,membership_availability_start,membership_availability_end,membership_available_online,membership_duration,membership_duration_type,membership_auto_renew,membership_pro_rata_start,membership_renew_duration,membership_renew_term,membership_renew_before,membership_renew_before_term,membership_renew_price,membership_price,membership_price_term,membership_income_category,membership_registration_fee,membership_club_tax,membership_billing_cycle,default_payment_method,tmp_default_payment_method,membership_creation_date,membership_last_edit_date,membership_invoice_creation_term,access_times
count,36.000000,36,36,21,21,36,36.000000,36,36,36,33.0,33,33.000000,33,36.000000,36.000000,36,36,34.0,36,35,36,36,36,36,36,1
unique,NaN,35,1,1,1,1,NaN,1,2,2,NaN,1,NaN,1,NaN,NaN,2,2,NaN,2,1,1,1,4,7,3,1
top,NaN,CUOTA PRO FUNDADOR + YOGA,default,1970-01-01,1970-01-02,False,NaN,months,True,True,NaN,months,NaN,days,NaN,NaN,monthly,memberships,NaN,"{'tax_id': '1', 'tax_name': 'IVA 21%', 'tax_percentage': 21}",monthly,directdebit_NL,direct_debit,2024-02-06,2026-02-22,1 weeks,"[{'day': 'Sunday', 'start_time': '06:00:00', 'end_time': '23:00:00'}, {'day': 'Monday', 'start_time': '06:00:00', 'end_time': '23:00:00'}, {'day': 'Tuesday', 'start_time': '06:00:00', 'end_time': '23:00:00'}, {'day': 'Wednesday', 'start_time': '06:00:00', 'end_time': '23:00:00'}, {'day': 'Thursday', 'start_time': '06:00:00', 'end_time': '23:00:00'}, {'day': 'Friday', 'start_time': '06:00:00', 'end_time': '23:00:00'}, {'day': 'Saturday', 'start_time': '06:00:00', 'end_time': '23:00:00'}]"
freq,NaN,2,36,21,21,36,NaN,36,33,28,NaN,33,NaN,33,NaN,NaN,35,35,NaN,34,35,36,36,24,14,18,1
mean,297305.388889,NaN,NaN,NaN,NaN,NaN,2.083333,NaN,NaN,NaN,1.0,NaN,6.515152,NaN,51.055556,55.888889,NaN,NaN,20.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,24705.212761,NaN,NaN,NaN,NaN,NaN,6.500000,NaN,NaN,NaN,0.0,NaN,0.667140,NaN,26.374170,21.780434,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,285036.000000,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN,1.0,NaN,4.000000,NaN,0.000000,0.000000,NaN,NaN,20.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,285066.500000,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN,1.0,NaN,6.000000,NaN,35.250000,41.500000,NaN,NaN,20.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,292082.500000,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN,1.0,NaN,7.000000,NaN,49.000000,52.000000,NaN,NaN,20.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,292091.250000,NaN,NaN,NaN,NaN,NaN,1.000000,NaN,NaN,NaN,1.0,NaN,7.000000,NaN,76.000000,76.000000,NaN,NaN,20.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
df_memberships.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2367 entries, 45988951 to 46764456
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   membership_id      2367 non-null   int64 
 1   contract_end_date  2367 non-null   object
dtypes: int64(1), object(1)
memory usage: 55.5+ KB


In [21]:
df_memberships.head(5)

,instance_id,member_id,membership_id,active,cancelled,contract_autorenewed,completed,paused,stopped,start_date,contract_start_date,contract_end_date,membership_name
instance_id,,,,,,,,,,,,,
9489382,9489382,45988423,285061,True,False,True,False,False,False,2019-09-17,2019-10-01,2026-06-30,CUOTA YOGA + GYM FAMILIAR
9489379,9489379,45988412,285059,False,True,True,False,False,True,2022-10-27,2022-11-01,2024-06-01,CUOTA YOGA + GYM
9489378,9489378,45988409,292085,True,False,True,False,False,False,2023-12-12,2024-01-01,2026-06-30,CUOTA TERCER FAMILIAR
9489377,9489377,45988406,292084,False,True,True,False,False,True,2022-11-08,2022-12-01,2025-05-07,CUOTA MENSUAL
9489376,9489376,45988405,285037,False,True,True,False,False,True,2023-02-20,2023-03-01,2024-05-29,CUOTA FAMILIAR


In [28]:
df_memberships.describe(include='all')

,instance_id,member_id,membership_id,active,cancelled,contract_autorenewed,completed,paused,stopped,start_date,contract_start_date,contract_end_date,membership_name
count,3.010000e+03,3.010000e+03,3010.000000,3010,3010,3010,3010,3010,3010,3010,3010,3010,3010
unique,NaN,NaN,NaN,2,2,2,2,2,2,934,102,221,19
top,NaN,NaN,NaN,False,True,True,False,False,True,2025-12-18,2026-01-01,2026-02-28,CUOTA MENSUAL
freq,NaN,NaN,NaN,1718,1734,2611,3001,3004,1703,186,285,1242,1505
mean,1.078162e+07,4.807392e+07,289251.796013,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,1.352740e+06,3.022779e+06,3797.035106,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,9.488900e+06,4.561678e+07,285036.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,9.489659e+06,4.598828e+07,285037.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
50%,1.018798e+07,4.598961e+07,292084.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
75%,1.210391e+07,5.018789e+07,292084.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
df_visits.head()

,id,club_id,member_id,device_id,check_in_timestamp,check_out_timestamp,status,status_message
id,,,,,,,,
112545937,112545937,79788,44229588,11667,1720623881000,1720623885000,ok,NaN
112544960,112544960,79788,44842160,11668,1720623580000,0,ok,NaN
112544879,112544879,79788,46888809,11667,1720623554000,1720623556000,ok,"Membresías: CUOTA FAMILIAR, CUOTA FAMILIAR"
112544557,112544557,79788,45988606,11667,1720623439000,1720623441000,ok,Membresías: CUOTA FAMILIAR
112543410,112543410,79788,45989782,11667,1720623041000,1720623042000,ok,Membresías: CUOTA FUNDADOR FAMILIAR


In [30]:
df_visits.info()

<class 'pandas.core.frame.DataFrame'>
Index: 218014 entries, 112545937 to 223494686
Data columns (total 8 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   id                   218014 non-null  int64 
 1   club_id              218014 non-null  int64 
 2   member_id            218014 non-null  int64 
 3   device_id            218014 non-null  int64 
 4   check_in_timestamp   218014 non-null  int64 
 5   check_out_timestamp  218014 non-null  int64 
 6   status               218014 non-null  object
 7   status_message       213357 non-null  object
dtypes: int64(6), object(2)
memory usage: 15.0+ MB


In [31]:
df_visits.describe(include='all')

,id,club_id,member_id,device_id,check_in_timestamp,check_out_timestamp,status,status_message
count,2.180140e+05,218014.0,2.180140e+05,218014.000000,2.180140e+05,2.180140e+05,218014,213357
unique,NaN,NaN,NaN,NaN,NaN,NaN,3,47
top,NaN,NaN,NaN,NaN,NaN,NaN,ok,Membresías: CUOTA MENSUAL
freq,NaN,NaN,NaN,NaN,NaN,NaN,216515,98751
mean,1.619064e+08,79788.0,4.720416e+07,11667.000018,1.746360e+12,1.394434e+12,NaN,NaN
std,3.197252e+07,0.0,2.311157e+06,0.004283,1.430041e+10,7.007022e+11,NaN,NaN
min,1.125413e+08,79788.0,4.422945e+07,11667.000000,1.720622e+12,0.000000e+00,NaN,NaN
25%,1.342733e+08,79788.0,4.598803e+07,11667.000000,1.733938e+12,1.725032e+12,NaN,NaN
50%,1.572827e+08,79788.0,4.598913e+07,11667.000000,1.746084e+12,1.740413e+12,NaN,NaN
75%,1.893181e+08,79788.0,4.743520e+07,11667.000000,1.759296e+12,1.756401e+12,NaN,NaN


2. EDA: Transformación y limpieza.

En esta etapa, se preparan y mejoran los datos para que sean más fáciles de analizar. La transformación y limpieza incluyen la detección y corrección de errores, la normalización o estandarización de variables, y la transformación de datos cuando sea necesario.
Técnicas:
-Eliminación de datos protegidos por RGPD y datos irrelevantes para el análisis.
-Corrección de tipos de datos: Convertir las variables mal etiquetadas (por ejemplo, transformar variables categóricas en variables de tipo adecuado).
-Eliminación de datos duplicados.
-Manejo de valores nulos: Determinar si es necesario eliminar, imputar o dejar sin cambios los valores faltantes.
-Normalización y estandarización: Ajustar los datos para que tengan una escala consistente, lo que es útil para ciertas técnicas de análisis.
-Transformaciones: Aplicar transformaciones logarítmicas o de potencias para corregir distribuciones sesgadas.


CLUB MEMBERS
Las tareas de transformación y limpieza son:
 -fijar el índice al "member_id" que es la id único de los clientes
 -eliminar campos por protección de datos (firstname, lastname, email, mobile, phone)
 -eliminar campos no relevantes (club_id, external_id, is_pro, active, lang, country, rfid_tag, early_booking_access, user_id, formatted_address)
 -se convierte a tipo date las fechas (birthday está como string, membership_since y timestamp_edit están en milisegundos)
 -timestamp_edit (tiempo en que se actualiza el registro), se elimina pero se crean dos campos, uno con timestamp_edit_date, y otro como 
 datetime (timestamp_edit_datetime): timestamp_edit_date si es igual contract_end_date se trata de un baja, y timestamp_edit_datetime
 su valor máximo determina la fecha y hora de la última modificación
- se eliminan los empleados de la tabla de miembros
- se eliminan los miembros que no tienen una membresía asociada (clientes no activos cuando se hizo la carga inicial del CRM en el año 2023)


 MEMBERSHIP DEFINITION
 Las tareas de transformación y limpieza son:
-se fija el índice a membership_id
-se eliminan todos los campos salvo membership_id, membership_name, membership_price
al mantenerse sólo el membership_price los ingresos calculados como suma de membership_price indicarán la "base de ingresos"
proforma, pero no la facturación real ya que esa implicaría hacer prórrata de la cuota y sumar también la matrícula 
la facturación no es el objetivo del análisis
-se cambia el tipo de datos de datos membership_name (tipo object) a string y el tipo de membership_price de tipo int a tipo float

MEMBERSHIP INSTANCES
Las tareas de transformación y limpieza son:
-fijar el índice a instance_id
-los clientes sólo tienen una membresía al mismo tiempo (el gimnasio es mono producto), pero las membresías
han cambiado durante el tiempo; por ello, se elige para cada cliente únicamente la membresía contract_end_date más reciente,
-se seleccionan únicamente las columnas relevantes para el análisis: member_id, membership_id, contract_end_date
-se convierte el tipo de datos contract_end_date de tipo string a tipo date
-se indexa por member_id

CLUB VISITS
Las tareas de transformación y limpieza son:
-se fija como índice el campo 'id'
-se seleccionan los campos relevantes (id, member_id, check_in_timestamp, check_out_timestamp, status)
-se consideran sólo visitas asociadas a los miembros, ya que es lo que se quiere caracterizar (no las de empleados)
-se consideran las visitas con status ok (las que se corresponden con el evento de entrada o salida con éxito, sino sería intentos de entrada/salida sería parte de otro análisis)
-se convierte check_in_timestamp en formato date para que sirva como eje temporal 
-se convierte check_in_timestamp en formato datetime
-se convierte check_out_timestamp en formato datetime
-se eliminan los días que se cerró el gimnasio
-se crea una columna adicional que es la duración de la visita en minutos, tendrá valor vacío si no existe check_out_timestamp
-se eliminan las visitas de duración menores a 5 minutos, las de menor duración se entienden que son anómalas
-se deduplican accesos que han ocurrido en menos de 10 min (umbral para deduplicar), se elige el último que se entiende que ha sido el exitoso
-se elimina la última visita de cada baja al gimnasio, tanto en la tabla de visitas como en la df_visits_minute_details que suele ser la visita presencial para comunicarla la baja, ya que no refleja un acceso real al gimnasio 
-una vez depuradas las visitas, es crítico calcular la concurrencia al centro de en cada minuto de cada visita de cada cliente, puesto que esto determina su experiencia.Para ello:
    - es necesario saber cuándo acaba y termina una visita, como muchas visitas no tienen checkou, se estima un check_out  aún cuando cliente no lo haya hecho. Para ello se estima la duración de la vista en función del promecio de las visitas de los últimos 30 días naturales anteriores  de ese cliente y si no se toma un promedio del centro que son 75 min. Esa estimación es auxiliar sólo para calcular el check_out ajustado, no es que la reemplace cuando sea cero, para no contaminar otros análisis. 
    -se realiza un dataframe auxiliar (df_datetimemin_visits) con todos dos datetimemin de apertura del centro y que contengan el número de visitas concurrentes en cada minuto; de esta forma el cálculo la concurrencia en cada minuto se realiza un única vez y no una por cada visita por cada miembro
    -a partir del dataframe anterior, se realiza un datafrema auxiliar que para id de visita (df_visits_minute_detail), id de cliente, minuto de la visita, determina la concurrencia (con cuántos clientes coincidió incluyendo él)

CONSOLIDACIÓN DE DATASETS

Para mayor facilidad en en análisis, se consolida la información en dos datasets, para que cada uno contenga toda la información
-Clientes (df_clients): dataset indexado por member_id que consolida por member_id, la información de los datasets: Club Members, Membership Definition, Membership Instances
-Visitas (df_visits): dataset indexado por id (visita) que consolida la información de los dataset de Clientes: fecha de nacimiento, sexo, fecha de alta, fecha de baja, membresía
-Detalle de al visita (df_visits_minute_detail), que contiene la concurrencia por minuto de cada visita de cada miembro

GENERACIÓN DE FICHEROS DE DATOS
Se generan tres ficheros de datos csv, con los dataset de clientes, visitas, y detalle de visitas. Estos serían el origen de datos de PowerBI.

(VER CÓDIDO PYTHON ANEXO)

In [24]:
df_members.sample()

,member_id,firstname,lastname,active,is_pro,gender,email,member_since,timestamp_edit,country,rfid_tag,club_id,registration_date,lang,original_member_id,early_booking_access,user_id,vat_number,birthday,zip,street,place,mobile,street_extra,formatted_address,club_member_id,phone
member_id,,,,,,,,,,,,,,,,,,,,,,,,,,,
47200707,47200707,Alba,Pascual Huelmo,True,True,f,albaph61@gmail.com,1719878400000,1766346677477,ES,NaN,79788,1719936689,es,0,True,29299354.0,54032432G,2006-03-06,28914,Calle De Alcobendas 7,Leganés,644493946,"7A, 1A","C. de Alcobendas, 7, 28914 Leganés, Madrid, Spain",54032432G,NaN


In [25]:
#CLUB MEMBERS - Transformación y Limpieza

#se seleccionan únicamente las columnas necesarias para el análisis

df_members.drop(columns=df_members.columns.difference(['member_id', 'gender', 'zip', 'birthday', 'member_since', 'timestamp_edit']), inplace=True)

#se eliminan los empleados de la tabla de miembros
empleados_id=list(df_employees['member_id'])
df_members=df_members[df_members.member_id.isin(empleados_id)==False]

#se eliminan los miembros que no tienen una membresía asociada (carga histórica antes del CRM), esto también elimina friends&family (sin membresía)
miembros_membresía=list(df_memberships["member_id"])
df_members=df_members[df_members.member_id.isin(miembros_membresía)==True]

#se convierten el tipo de datos de birthday (string) en tipo date, generándose una variable adicional "birthday_c"
#función que convierte un string en valor tipo date 
import datetime as dt
def convertir_date(fecha_a_convertir):
    try:
        fecha_str_split=str(fecha_a_convertir).split('-')
        return dt.date(int(fecha_str_split[0]),int((fecha_str_split[1])), int(fecha_str_split[2]))
    except ValueError:
        return None
    except TypeError:
        return None
    except IndexError:
        return None
    
#se convierte la fecha de nacimiento en formato date
df_members['birthday']=df_members['birthday'].apply(lambda x: None if x==None else convertir_date(x))

#se convierte del tipo de datos de fecha de alta de milisegundos a tipo date
df_members['member_since']=df_members['member_since'].apply(lambda x: None if x==None else dt.datetime.fromtimestamp(x/1000).date())

#se convierte la variable "timestamp_edit_date" en tipo date 
df_members['timestamp_edit_date']=df_members['timestamp_edit'].apply(lambda x: None if x==None else dt.datetime.fromtimestamp(x/1000).date())

#se convierte la variable "timestamp_edit_datetime" en tipo datetime para mantener la fecha de la última actualización
df_members['timestamp_edit_datetime']=df_members['timestamp_edit'].apply(lambda x: None if x==None else dt.datetime.fromtimestamp(x/1000))

#se elimina la variable "timestamp" en milsegundos
df_members.drop("timestamp_edit", axis=1, inplace=True)

#se fija el índice como "member_id"
df_members.set_index('member_id', inplace=True)

In [26]:
df_members.head()

,gender,member_since,birthday,zip,timestamp_edit_date,timestamp_edit_datetime
member_id,,,,,,
45986920,m,2019-07-15,1985-05-17,28944,2025-12-21,2025-12-21 20:00:09.058
45986919,f,2018-09-26,1983-09-13,28942,2025-12-21,2025-12-21 20:02:07.121
45986916,f,2019-09-12,1972-02-28,28914,2025-12-21,2025-12-21 20:00:23.756
45986912,f,2021-05-08,1994-11-12,28914,2025-09-15,2025-09-15 09:58:44.200
45824256,m,2024-03-15,1991-01-01,NaN,2024-06-01,2024-06-01 13:38:00.022


In [27]:
# MEMBERSHIP DEFINITION- Transformación y Limpieza

#se seleccionan sólo las columnas relevantes
df_memberships_definition.drop(columns=df_memberships_definition.columns.difference(['membership_id', 'membership_name', 'membership_price']), inplace=True)

#se cambia el tipo de membership_name a string
df_memberships_definition['membership_name']=df_memberships_definition['membership_name'].astype(str)

#se cambia el tipo de membership_price a int
df_memberships_definition['membership_price']=df_memberships_definition['membership_price'].astype(str)

#se fija el índice como "membership_id"
df_memberships_definition.set_index('membership_id', inplace=True)

In [28]:
df_memberships_definition.sample()

,membership_name,membership_price
membership_id,,
292089,CUOTA TERCER FAMILIAR FUNDADOR,33


In [29]:
df_memberships_definition.info(10)

<class 'pandas.core.frame.DataFrame'>
Index: 36 entries, 292096 to 292084
Data columns (total 2 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   membership_name   36 non-null     object
 1   membership_price  36 non-null     object
dtypes: object(2)
memory usage: 864.0+ bytes


In [30]:
#MEMBERSHIP INSTANCES- Transformación y Limpieza

#se seleccionan únicamente las columnas necesarias para el análisis
df_memberships.drop(columns=df_memberships.columns.difference(['member_id', 'membership_id', 'contract_end_date']), inplace=True)

#se convierte contract_end_date en formato date
df_memberships['contract_end_date'] = (df_memberships['contract_end_date'].apply(lambda x: None if x is None else convertir_date(x)))

#se elige la membresía más reciente
df_memberships=(df_memberships.sort_values("contract_end_date", ascending=False).groupby("member_id").head(1))

#se fija el índice como "member_id"
df_memberships.set_index('member_id', inplace=True)


In [31]:
df_memberships.sample()

,membership_id,contract_end_date
member_id,,
45988182,292084,2026-06-30


In [32]:
#CLUB VISITS- Transformación y Limpieza

#se fija el índice como "instance_id"
df_visits.set_index('id', inplace=True)

#se seleccionan únicamente las columnas necesarias para el análisis
df_visits.drop(columns=df_visits.columns.difference(['member_id', 'check_in_timestamp', 'check_out_timestamp','status']), inplace=True)

#se eliminan de la tabla de visitas las que no estén asociadas a un miembro 
empleados=list(df_members.index)
df_visits=df_visits[df_visits.member_id.isin(empleados)==True]

#se seleccionan de las tablas de visitas sólo las visitas con éxito (status=ok)
df_visits=df_visits[df_visits['status']=='ok']
df_visits.drop(['status'], axis=1, inplace=True)

#se convierte check_in_timestamp en formato date para que sirva como eje temporal 
df_visits['check_in_timestamp_date']=df_visits['check_in_timestamp'].apply(lambda x: None if x==None else dt.datetime.fromtimestamp(x/1000).date())

#se convierte check_in_timestamp en formato datetime
df_visits['check_in_timestamp']=df_visits['check_in_timestamp'].apply(lambda x: None if x==None else dt.datetime.fromtimestamp(x/1000))

#se convierte check_out_timestamp en formato datetime
df_visits['check_out_timestamp']=df_visits['check_out_timestamp'].apply(lambda x: None if (x==None or x==0) else dt.datetime.fromtimestamp(x/1000))

# se eliminan los días que se cerró el gimnasio
dias_purga_visitas = ['2026-03-22', '2026-04-01', '2026-04-02', '2026-04-03', '2026-04-09']
indices_dias_purga = df_visits[df_visits['check_in_timestamp_date'].astype(str).isin(dias_purga_visitas)].index
df_visits.drop(index=indices_dias_purga, inplace=True)


In [33]:
df_visits.head()

,member_id,check_in_timestamp,check_out_timestamp,check_in_timestamp_date
id,,,,
112544879,46888809,2024-07-10 16:59:14,2024-07-10 16:59:16,2024-07-10
112544557,45988606,2024-07-10 16:57:19,2024-07-10 16:57:21,2024-07-10
112543410,45989782,2024-07-10 16:50:41,2024-07-10 16:50:42,2024-07-10
112542527,45989590,2024-07-10 16:44:31,NaT,2024-07-10
112542484,45989642,2024-07-10 16:44:10,NaT,2024-07-10


In [34]:

#se deduplican accesos que han ocurrido en menos de 10 min (umbral para deduplicar), se elige el último que
#se entiende que ha sido el exitoso
df_visits_sorted=df_visits.sort_values(["member_id", "check_in_timestamp"])
umbral_dupl=pd.Timedelta(minutes=10)
df_visits_sorted['diferencia_posterior']=-df_visits_sorted.groupby('member_id')['check_in_timestamp'].diff(periods=-1)
visitas_dedup= df_visits_sorted["diferencia_posterior"].isna() | (df_visits_sorted["diferencia_posterior"] > umbral_dupl)
df_visits_sorted=df_visits_sorted[visitas_dedup]
df_visits_sorted.drop(columns=['diferencia_posterior'], inplace=True)
df_visits=df_visits_sorted


In [35]:
#se crea una columna adicional que es la duración de la visita en minutos, tendrá valor vacío si no existe check_out_timestamp
df_visits['duration_min'] = (df_visits['check_out_timestamp'] - df_visits['check_in_timestamp']).dt.total_seconds() .floordiv(60) .astype('Int64')  

In [36]:
#se eliminan las visitas de duración menores a 5 minutos
df_visits=df_visits[(df_visits['duration_min'] >= 5) | (df_visits['duration_min'].isna())]

In [37]:
df_members.head(5)

,gender,member_since,birthday,zip,timestamp_edit_date,timestamp_edit_datetime
member_id,,,,,,
45986920,m,2019-07-15,1985-05-17,28944,2025-12-21,2025-12-21 20:00:09.058
45986919,f,2018-09-26,1983-09-13,28942,2025-12-21,2025-12-21 20:02:07.121
45986916,f,2019-09-12,1972-02-28,28914,2025-12-21,2025-12-21 20:00:23.756
45986912,f,2021-05-08,1994-11-12,28914,2025-09-15,2025-09-15 09:58:44.200
45824256,m,2024-03-15,1991-01-01,NaN,2024-06-01,2024-06-01 13:38:00.022


In [38]:
df_visits.head()

,member_id,check_in_timestamp,check_out_timestamp,check_in_timestamp_date,duration_min
id,,,,,
154513711,45986912,2025-04-14 22:06:14,NaT,2025-04-14,<NA>
154668532,45986912,2025-04-15 17:58:36,2025-04-15 19:14:11,2025-04-15,75
154697920,45986912,2025-04-15 19:14:13,NaT,2025-04-15,<NA>
154925506,45986912,2025-04-16 19:44:30,2025-04-16 20:45:20,2025-04-16,60
155112248,45986912,2025-04-17 20:36:45,2025-04-17 21:32:40,2025-04-17,55


In [39]:
#para calcular la concurrencia del centro en cada momento es necesario estimar un check_out aún cuando el cliente no lo haya hecho
#para ello se estima la duración de la vista en función del promecio de las visitas de los últimos 30 días naturales anteriores  de ese cliente y si no se toma un promedio del centro que son 75 min

import duckdb

# Registramos el dataframe actual en DuckDB
df_v = df_visits.reset_index()

df_visits = duckdb.query("""
    WITH historia_30d AS (
        SELECT 
            *,
            -- Calculamos el promedio de los últimos 30 días para ese miembro
            AVG(duration_min) OVER (
                PARTITION BY member_id 
                ORDER BY CAST(check_in_timestamp AS TIMESTAMP)
                RANGE BETWEEN INTERVAL '30 days' PRECEDING AND INTERVAL '1 SECOND' PRECEDING
            ) as promedio_movil
        FROM df_v
    )
    SELECT 
        *,
        -- Si no hay check_out, sumamos el promedio móvil (o 75 si es nulo) al check_in
        COALESCE(
            check_out_timestamp, 
            check_in_timestamp + INTERVAL (COALESCE(promedio_movil, 75)) MINUTE
        ) as check_out_ajustado
    FROM historia_30d
""").df()

# Limpieza final: redondeo a segundos como tenías antes
df_visits['check_out_ajustado'] = df_visits['check_out_ajustado'].dt.floor('s')

df_visits.drop(columns=['promedio_movil'], inplace=True)

df_visits.rename(columns={'index': 'id'}, inplace=True)
df_visits.set_index('id', inplace=True)


In [40]:
df_visits.head()

,member_id,check_in_timestamp,check_out_timestamp,check_in_timestamp_date,duration_min,check_out_ajustado
id,,,,,,
230403681,45988573,2026-02-28 15:10:59,2026-02-28 16:59:26,2026-02-28,108,2026-02-28 16:59:26
231317698,45988573,2026-03-03 19:17:05,2026-03-03 21:14:35,2026-03-03,117,2026-03-03 21:14:35
231984176,45988573,2026-03-05 16:31:58,2026-03-05 17:47:03,2026-03-05,75,2026-03-05 17:47:03
232369114,45988573,2026-03-06 17:19:18,2026-03-06 18:40:45,2026-03-06,81,2026-03-06 18:40:45
233062861,45988573,2026-03-09 17:19:59,2026-03-09 18:37:35,2026-03-09,77,2026-03-09 18:37:35


In [41]:
#hacemos una tabla (dataframe) tipo datetimemin para calcular la visitas concurrentes en cada minuto
#esta tabla va desde el mínimo y máximo de check_in_datetime de las visitas

#lo primero es hacer dos medidas auxiliares de redondeo a minutos de check_in y check_out para coincida con la escala de datetimemin
df_visits['check_out_ajustado_min'] = df_visits['check_out_ajustado'].dt.floor('min')
df_visits['check_in_timestamp_min'] = df_visits['check_in_timestamp'].dt.floor('min')

#no consideramos las entradas después de las 23 horas y las salidas después de las 23 horas las igualamos a las 23
df_visits = df_visits[df_visits['check_in_timestamp_min'].dt.hour < 23]

mask_tarde = df_visits['check_out_ajustado_min'] > (df_visits['check_in_timestamp_min'].dt.normalize() + pd.Timedelta(hours=23))

df_visits.loc[mask_tarde, 'check_out_ajustado_min'] = df_visits.loc[mask_tarde, 'check_in_timestamp_min'].dt.normalize() + pd.Timedelta(hours=23)



In [42]:
#se convierte check_out_ajustado_min en formato date para que se puede comparar mejor con los eje temporal de días
df_visits['check_out_timestamp_date'] = df_visits['check_out_ajustado_min'].dt.date

In [43]:
df_visits.sample()

,member_id,check_in_timestamp,check_out_timestamp,check_in_timestamp_date,duration_min,check_out_ajustado,check_out_ajustado_min,check_in_timestamp_min,check_out_timestamp_date
id,,,,,,,,,
189912337,45988361,2025-10-02 22:42:05,NaT,2025-10-02,<NA>,2025-10-02 23:46:05,2025-10-02 23:00:00,2025-10-02 22:42:00,2025-10-02


In [44]:
#para ser explotado en BI, sólo consideramos las columnas relevantes ya ajustadas

max_check_out=df_visits['check_out_timestamp'].max()

df_visits.drop(columns=df_visits.columns.difference(['id', 'member_id', 'check_in_timestamp_date', 
                                                               'check_in_timestamp_min', 'check_out_timestamp_date', 'check_out_ajustado_min', 'duration_min']), inplace=True)

df_visits=df_visits.rename(columns={'check_out_ajustado_min': 'check_out_timestamp',  'check_in_timestamp_min': 'check_in_timestamp'})

In [45]:
#reordenamos columnas
df_visits = df_visits[['member_id', 'check_in_timestamp', 'check_in_timestamp_date', 'check_out_timestamp', 'check_out_timestamp_date', 'duration_min']]

In [46]:
#hacemos la tabla datetimemin_visits
min_time = df_visits['check_in_timestamp'].min()
max_time = max(df_visits['check_in_timestamp'].max(), max_check_out)

df_datetimemin_visits = pd.DataFrame(index=pd.date_range(start=min_time, end=max_time, freq='min')).rename_axis('datetimemin')

In [47]:
#limitamos la tabla al horario de apertura
df_datetimemin_visits = df_datetimemin_visits.between_time('06:00', '23:00')

In [48]:

#  Contamos entradas y salidas por minuto
df_visits = df_visits.sort_values(['check_in_timestamp'])
entradas = df_visits['check_in_timestamp'].value_counts()
salidas = df_visits['check_out_timestamp'] .value_counts()

# Alineamos con la tabla maestra (usando el índice filtrado)
df_datetimemin_visits['entradas'] = entradas.reindex(df_datetimemin_visits.index, fill_value=0)
df_datetimemin_visits['salidas'] = salidas.reindex(df_datetimemin_visits.index, fill_value=0)

# se calculan las entradas y salidas acumuladas, reiniciando cada día
df_datetimemin_visits = df_datetimemin_visits.sort_index()
df_datetimemin_visits['entradas_acum'] = df_datetimemin_visits.groupby(pd.Grouper(freq='D'))['entradas'].cumsum()
df_datetimemin_visits['salidas_acum'] = df_datetimemin_visits.groupby(pd.Grouper(freq='D'))['salidas'].cumsum()

# la diferencia entre entradas y salidas diarias es la concurrencia
df_datetimemin_visits['concurrencia'] = df_datetimemin_visits['entradas_acum'] - df_datetimemin_visits['salidas_acum']

# se eliminan los días cerrados del gimnasio
dias_purga_visitas = ['2026-03-22', '2026-04-01', '2026-04-02', '2026-04-03', '2026-04-09']
indices_c = df_datetimemin_visits[df_datetimemin_visits.index.to_series().dt.date.astype(str).isin(dias_purga_visitas)].index
df_datetimemin_visits.drop(index=indices_c, inplace=True)


In [49]:
df_datetimemin_visits.sample()

,entradas,salidas,entradas_acum,salidas_acum,concurrencia
datetimemin,,,,,
2026-03-24 07:32:00,0,0,38,16,22


In [50]:
df_visits.sample()

,member_id,check_in_timestamp,check_in_timestamp_date,check_out_timestamp,check_out_timestamp_date,duration_min
id,,,,,,
139080352,47990293,2025-01-16 08:27:00,2025-01-16,2025-01-16 10:19:00,2025-01-16,111


In [51]:
#vamos a realizar una tabla que tenga la concurrencia de cada minuto visita de cada cliente
#permitirá determinar al nivel visita y/o cliente los minutos - ocupación y determinar cuántos minutos tuvieron de baja-media-alta ocupación
#y también explorar relaciones de esta ocupación con el churn y otras variables
#nuevamente se realiza en sql porque con pandas el tiempo que tardaban era de muchos minutos 

import duckdb

# se liberan índices para que SQL los vea como
df_c = df_datetimemin_visits.reset_index()
df_v = df_visits.reset_index()

# consulta en sql
df_visits_minute_detail = duckdb.query("""
    SELECT 
        v.id as visit_id, 
        v.member_id, 
        c.datetimemin, 
        c.concurrencia
    FROM df_v v
    JOIN df_c c 
        ON c.datetimemin BETWEEN v.check_in_timestamp AND v.check_out_timestamp
""").df()

df_visits_minute_detail.set_index('visit_id', inplace=True)

#para facilitar la relación se añade una columna con el día de la visita
df_visits_minute_detail['date']=df_visits_minute_detail['datetimemin'].dt.date


In [52]:
df_visits_minute_detail.sample()

,member_id,datetimemin,concurrencia,date
visit_id,,,,
175557530,45989167,2025-08-03 19:51:00,17,2025-08-03


In [53]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
fecha_buscada = '2026-04-05'
df_dia = df_visits[df_visits['check_in_timestamp'].dt.date.astype(str) == fecha_buscada]
df_dia

,member_id,check_in_timestamp,check_in_timestamp_date,check_out_timestamp,check_out_timestamp_date,duration_min
id,,,,,,
241228696,45987750,2026-04-05 06:00:00,2026-04-05,2026-04-05 07:33:00,2026-04-05,92
241229499,45988448,2026-04-05 06:32:00,2026-04-05,2026-04-05 08:10:00,2026-04-05,97
241230345,52567395,2026-04-05 07:00:00,2026-04-05,2026-04-05 08:23:00,2026-04-05,82
241230439,45988400,2026-04-05 07:04:00,2026-04-05,2026-04-05 08:05:00,2026-04-05,60
241230701,45988919,2026-04-05 07:11:00,2026-04-05,2026-04-05 08:59:00,2026-04-05,107
241231149,54707381,2026-04-05 07:25:00,2026-04-05,2026-04-05 08:23:00,2026-04-05,57
241231264,45989148,2026-04-05 07:27:00,2026-04-05,2026-04-05 08:34:00,2026-04-05,66
241231930,53604441,2026-04-05 07:43:00,2026-04-05,2026-04-05 09:01:00,2026-04-05,77
241232103,45987964,2026-04-05 07:47:00,2026-04-05,2026-04-05 09:23:00,2026-04-05,95


In [50]:
pd.set_option('display.max_rows', None)
df_filtrado = df_datetimemin_visits.loc['2026-04-05']
df_filtrado

,entradas,salidas,entradas_acum,salidas_acum,concurrencia
datetimemin,,,,,
2026-04-05 06:00:00,1,0,1,0,1
2026-04-05 06:01:00,0,0,1,0,1
2026-04-05 06:02:00,0,0,1,0,1
2026-04-05 06:03:00,0,0,1,0,1
2026-04-05 06:04:00,0,0,1,0,1
2026-04-05 06:05:00,0,0,1,0,1
2026-04-05 06:06:00,0,0,1,0,1
2026-04-05 06:07:00,0,0,1,0,1
2026-04-05 06:08:00,0,0,1,0,1


In [51]:
df_datetimemin_visits.tail(10)

,entradas,salidas,entradas_acum,salidas_acum,concurrencia
datetimemin,,,,,
2026-05-18 13:15:00,1,0,178,142,36
2026-05-18 13:16:00,1,0,179,142,37
2026-05-18 13:17:00,0,0,179,142,37
2026-05-18 13:18:00,0,0,179,142,37
2026-05-18 13:19:00,1,0,180,142,38
2026-05-18 13:20:00,0,2,180,144,36
2026-05-18 13:21:00,1,0,181,144,37
2026-05-18 13:22:00,0,0,181,144,37
2026-05-18 13:23:00,0,0,181,144,37


In [52]:
df_visits.sample()

,member_id,check_in_timestamp,check_in_timestamp_date,check_out_timestamp,check_out_timestamp_date,duration_min
id,,,,,,
189815918,45988883,2025-10-02 17:49:00,2025-10-02,2025-10-02 19:16:00,2025-10-02,<NA>


In [54]:
#CONSOLIDACIÓN DE DATASETS

#DATASET CLIENTES
#se crea un dataset nuevo, clientes, consolidando los dataset Club Members, Membership Definition, Membership Instances
#primero se consolida Membership Instances y Membership Definition, añadiendo a Membership Instances la información nombre, y precio de la membresía

df_memberships = df_memberships.reset_index()
df_members= df_members.reset_index()
df_memberships= df_memberships.merge( df_memberships_definition, on="membership_id",how="left")
df_clients = df_members.merge(df_memberships, on="member_id", how="left")
df_clients.set_index("member_id", inplace=True)


In [55]:
df_members.head()

,member_id,gender,member_since,birthday,zip,timestamp_edit_date,timestamp_edit_datetime
0,45986920,m,2019-07-15,1985-05-17,28944,2025-12-21,2025-12-21 20:00:09.058
1,45986919,f,2018-09-26,1983-09-13,28942,2025-12-21,2025-12-21 20:02:07.121
2,45986916,f,2019-09-12,1972-02-28,28914,2025-12-21,2025-12-21 20:00:23.756
3,45986912,f,2021-05-08,1994-11-12,28914,2025-09-15,2025-09-15 09:58:44.200
4,45824256,m,2024-03-15,1991-01-01,NaN,2024-06-01,2024-06-01 13:38:00.022


In [56]:
df_members.head()

,member_id,gender,member_since,birthday,zip,timestamp_edit_date,timestamp_edit_datetime
0,45986920,m,2019-07-15,1985-05-17,28944,2025-12-21,2025-12-21 20:00:09.058
1,45986919,f,2018-09-26,1983-09-13,28942,2025-12-21,2025-12-21 20:02:07.121
2,45986916,f,2019-09-12,1972-02-28,28914,2025-12-21,2025-12-21 20:00:23.756
3,45986912,f,2021-05-08,1994-11-12,28914,2025-09-15,2025-09-15 09:58:44.200
4,45824256,m,2024-03-15,1991-01-01,NaN,2024-06-01,2024-06-01 13:38:00.022


In [57]:

#se eliminand los ids cuyo contract_end_date coincide con los días de purga 
#los días de purga se dieron de baja de bajas masiva ids que estaban activos en el sistema por error
df_members.set_index('member_id', inplace=True)
días_purga=['28-05-2024', '29-05-2024','07-05-2025']
días_purga=list(map(lambda x: dt.datetime.strptime(x, "%d-%m-%Y").date(), días_purga))
members_id_purgados= df_clients[df_clients['contract_end_date'].isin(días_purga)].index.unique()
df_clients=df_clients[df_clients.index.isin(members_id_purgados)==False]

In [58]:
#introduczo dos valores adicionales para simplificar el análisis de cohortes, una la cohorte de alta y otra la cohorte de baja
df_clients['cohorte_alta']=df_clients['member_since'].apply(lambda x: None if x==None else dt.date(x.year, x.month, 1))

#la cohorte de baja se pone a null si el cliente no se ha dado de baja, el cliente se ha dado de baja cuando timestamap_edit es igual a contract_end_date
churn_clients = df_clients[
    df_clients["timestamp_edit_date"]
    == df_clients["contract_end_date"]
].index

df_clients['cohorte_baja']=None

df_clients.loc[churn_clients, "cohorte_baja"] = df_clients['contract_end_date'].apply(lambda x: None if x==None else dt.date(x.year, x.month, 1))

In [59]:
#se elimina la visita de cada miembro que coincide con la fecha de alta del mismo, ya que no refleja un acceso real al gimnasio
#tanto de la tabla df_visits como de df_visits_minute_detail para que no afecte al análisis de concurrencia y visitas por cliente
df_visits = df_visits.join(
    df_clients["member_since"],
    on="member_id"
)
df_visits = df_visits[
    df_visits["check_in_timestamp_date"] != df_visits["member_since"]
].drop(columns="member_since")

df_visits_minute_detail = df_visits_minute_detail.join(
    df_clients["member_since"],
    on="member_id"
)
df_visits_minute_detail = df_visits_minute_detail[
    df_visits_minute_detail["datetimemin"].dt.date != df_visits_minute_detail["member_since"]
].drop(columns="member_since")

In [60]:
#se elimina la última visita de cada baja al gimnasio, que suele ser la visita presencial para comunicarla la baja, ya que no refleja un acceso real al gimnasio
# Miembros considerados baja
#se hace tanto de la tabla df_visits como de df_visits_minute_detail para que no afecte al análisis de concurrencia y visitas por cliente

df_visits = df_visits.sort_values(
    ["member_id", "check_in_timestamp_date"]
)

idx_last_visit = (
    df_visits
    .groupby("member_id")["check_in_timestamp_date"]
    .idxmax()
)

last_visits = df_visits.loc[idx_last_visit]

idx_remove = last_visits[
    last_visits["member_id"].isin(churn_clients)
].index

df_visits = df_visits.drop(index=idx_remove)

df_visits_minute_detail = df_visits_minute_detail.sort_values(
    ["member_id", "datetimemin"]
)   
idx_last_visit_minute = (
    df_visits_minute_detail    .groupby("member_id")["datetimemin"]
    .idxmax()
)
last_visits_minute = df_visits_minute_detail.loc[idx_last_visit_minute] 
idx_remove_minute = last_visits_minute[
    last_visits_minute["member_id"].isin(churn_clients)
].index
df_visits_minute_detail = df_visits_minute_detail.drop(index=idx_remove_minute) 


In [61]:
df_members.head()

,gender,member_since,birthday,zip,timestamp_edit_date,timestamp_edit_datetime
member_id,,,,,,
45986920,m,2019-07-15,1985-05-17,28944,2025-12-21,2025-12-21 20:00:09.058
45986919,f,2018-09-26,1983-09-13,28942,2025-12-21,2025-12-21 20:02:07.121
45986916,f,2019-09-12,1972-02-28,28914,2025-12-21,2025-12-21 20:00:23.756
45986912,f,2021-05-08,1994-11-12,28914,2025-09-15,2025-09-15 09:58:44.200
45824256,m,2024-03-15,1991-01-01,NaN,2024-06-01,2024-06-01 13:38:00.022


In [62]:
df_clients.head()

,gender,member_since,birthday,zip,timestamp_edit_date,timestamp_edit_datetime,membership_id,contract_end_date,membership_name,membership_price,cohorte_alta,cohorte_baja
member_id,,,,,,,,,,,,
45986920,m,2019-07-15,1985-05-17,28944,2025-12-21,2025-12-21 20:00:09.058,292084,2026-06-30,CUOTA MENSUAL,52,2019-07-01,None
45986919,f,2018-09-26,1983-09-13,28942,2025-12-21,2025-12-21 20:02:07.121,292084,2026-06-30,CUOTA MENSUAL,52,2018-09-01,None
45986916,f,2019-09-12,1972-02-28,28914,2025-12-21,2025-12-21 20:00:23.756,285037,2026-06-30,CUOTA FAMILIAR,49,2019-09-01,None
45986912,f,2021-05-08,1994-11-12,28914,2025-09-15,2025-09-15 09:58:44.200,285037,2025-09-15,CUOTA FAMILIAR,49,2021-05-01,2025-09-01
45824256,m,2024-03-15,1991-01-01,NaN,2024-06-01,2024-06-01 13:38:00.022,292084,2024-06-01,CUOTA MENSUAL,52,2024-03-01,2024-06-01


In [63]:
df_visits.head()

,member_id,check_in_timestamp,check_in_timestamp_date,check_out_timestamp,check_out_timestamp_date,duration_min
id,,,,,,
154513711,45986912,2025-04-14 22:06:00,2025-04-14,2025-04-14 23:00:00,2025-04-14,<NA>
154668532,45986912,2025-04-15 17:58:00,2025-04-15,2025-04-15 19:14:00,2025-04-15,75
154697920,45986912,2025-04-15 19:14:00,2025-04-15,2025-04-15 20:29:00,2025-04-15,<NA>
154925506,45986912,2025-04-16 19:44:00,2025-04-16,2025-04-16 20:45:00,2025-04-16,60
155112248,45986912,2025-04-17 20:36:00,2025-04-17,2025-04-17 21:32:00,2025-04-17,55


In [64]:
df_members.head(5)

,gender,member_since,birthday,zip,timestamp_edit_date,timestamp_edit_datetime
member_id,,,,,,
45986920,m,2019-07-15,1985-05-17,28944,2025-12-21,2025-12-21 20:00:09.058
45986919,f,2018-09-26,1983-09-13,28942,2025-12-21,2025-12-21 20:02:07.121
45986916,f,2019-09-12,1972-02-28,28914,2025-12-21,2025-12-21 20:00:23.756
45986912,f,2021-05-08,1994-11-12,28914,2025-09-15,2025-09-15 09:58:44.200
45824256,m,2024-03-15,1991-01-01,NaN,2024-06-01,2024-06-01 13:38:00.022


In [65]:
df_clients.head(5)

,gender,member_since,birthday,zip,timestamp_edit_date,timestamp_edit_datetime,membership_id,contract_end_date,membership_name,membership_price,cohorte_alta,cohorte_baja
member_id,,,,,,,,,,,,
45986920,m,2019-07-15,1985-05-17,28944,2025-12-21,2025-12-21 20:00:09.058,292084,2026-06-30,CUOTA MENSUAL,52,2019-07-01,None
45986919,f,2018-09-26,1983-09-13,28942,2025-12-21,2025-12-21 20:02:07.121,292084,2026-06-30,CUOTA MENSUAL,52,2018-09-01,None
45986916,f,2019-09-12,1972-02-28,28914,2025-12-21,2025-12-21 20:00:23.756,285037,2026-06-30,CUOTA FAMILIAR,49,2019-09-01,None
45986912,f,2021-05-08,1994-11-12,28914,2025-09-15,2025-09-15 09:58:44.200,285037,2025-09-15,CUOTA FAMILIAR,49,2021-05-01,2025-09-01
45824256,m,2024-03-15,1991-01-01,NaN,2024-06-01,2024-06-01 13:38:00.022,292084,2024-06-01,CUOTA MENSUAL,52,2024-03-01,2024-06-01


In [66]:
df_clients.head()

,gender,member_since,birthday,zip,timestamp_edit_date,timestamp_edit_datetime,membership_id,contract_end_date,membership_name,membership_price,cohorte_alta,cohorte_baja
member_id,,,,,,,,,,,,
45986920,m,2019-07-15,1985-05-17,28944,2025-12-21,2025-12-21 20:00:09.058,292084,2026-06-30,CUOTA MENSUAL,52,2019-07-01,None
45986919,f,2018-09-26,1983-09-13,28942,2025-12-21,2025-12-21 20:02:07.121,292084,2026-06-30,CUOTA MENSUAL,52,2018-09-01,None
45986916,f,2019-09-12,1972-02-28,28914,2025-12-21,2025-12-21 20:00:23.756,285037,2026-06-30,CUOTA FAMILIAR,49,2019-09-01,None
45986912,f,2021-05-08,1994-11-12,28914,2025-09-15,2025-09-15 09:58:44.200,285037,2025-09-15,CUOTA FAMILIAR,49,2021-05-01,2025-09-01
45824256,m,2024-03-15,1991-01-01,NaN,2024-06-01,2024-06-01 13:38:00.022,292084,2024-06-01,CUOTA MENSUAL,52,2024-03-01,2024-06-01


In [67]:
df_visits.sample()

,member_id,check_in_timestamp,check_in_timestamp_date,check_out_timestamp,check_out_timestamp_date,duration_min
id,,,,,,
234343756,45987543,2026-03-12 19:44:00,2026-03-12,2026-03-12 21:21:00,2026-03-12,96


In [68]:
#se genera un fichero csv por cada dataframe añadiendo el timestamp de cuando se ha generado
df_clients.to_csv('Clients.csv')

In [69]:
df_visits.to_csv('Visits.csv')

In [70]:
#df_datetimemin_visits.to_csv('Datetimemin_visits.csv')

In [71]:
df_visits_minute_detail.to_csv('VisitsMinute_detail.csv')

DASHBOARD Y VISUALIZACIÓN DE LOS DATOS

El dashboard y visualización de los datos están realizados en PowerBI.

Como principio general, se realizan como medidas de PowerBI todas las variables que son calculadas y que varían en el tiempo (p.ej. número clientes activos, edad, antigüedad de un cliente, días desde la última visita etc ), incluso su categorización -en su caso- con segmentación dinámica (pej. segmentos de edad o permanencia). Las razones son:
-las medidas de PoweBI permiten calcular de forma eficiente variables dinámicas (ejecución bajo demanda, tablas temporales incluídas en el propio código DAX de la medida)
-se reduce la complejidad de la ETL: sólo se cargan Datasets con datos estáticos, no tablas con ejes temporales y valor de indicadores en éstos ejes

En la preparación del Dashboard en PowerBI se han seguido estas fases:
-ETL para importación y carga a través de PowerQuery; proceso muy sencillo ya que la limpieza y transformación está ya realizada en python en fases anteriores. Sólo carga del csv- encabezados promovidos, de las tablas "Clients", "Visits", "Visits_Minute_detail"
-Creación de dos tablas de eje temporal para poder realizar evolutivos: una de días "Calendario" para cálculos de kpis día, otra de minutos "CalendarioDateTimeMin" para poder hacer cálculos de KPIs-minutos (concurrencia), dos tablas para hacer el cálculos de cohortes ("Matriz_Altas_Cohortes", "Matriz_Bajas_Cohortes) que agrupan las fechas de la tabla Calendario por meses y las combinan con meses de antigüedad. Estas tablas además tienen la etiqueta del mes y día en formato numérico o texto y tienen un columna orden.
-activación de relaciones entre las tablas de datos: Clients, Visits, Visist_Minute_detail entre sí (relación a través de member_id), y de estas tablas con las tablas de tiempo Calendario, CalendarioDateTimeMin, Matriz_Altas_Cohortes y Matriz_Bajas_Cohortes
-estructuración del dashboard y medidas por página, agrupando medidas calculadas en tablas para mejor acceso (pej las métricas de Clientes por un lado, las de Visitas por otro etc)
-uso de la herramienta externa "Tabular Editor", para hacer los comparativos de un valor con respecto a referencias pasadas con Calculation Items (por ejemplo, comparativa con el mes anterior, año anterior etc). De esta forma, se realiza un Calculation Item por cada comparativo y no medida calculada para cada métrica y comparativo.

Dashboard:
El Dashboard de PowerBI se ha estructurado con estas páginas:
1. "Resumen ejecutivo": permite conocer de un vistazo el desempeño del negocio. Dos franjas. Primera con tarjetas de KPIs actuales en grande y comparativos abajo más pequeño: Número de Clientes, ARPU, Permanencia y MRR (debajo comparación con el 1 de mes, el mes pasado, y ese mes el año pasado al ser un negocio estacional, comparación tanto en valor absulto como relativo), y Churn (debajo comparación con el valor mes -1, mes -2, mes - 3). Segunda franja dos evolutivos para ver tendencias: 1 evolutivo mensual dos últimos años de Clientes a 1ro de mes, Altas, Bajas. 2. evolutivo mensual dos últimos años de ARPU, Churn
2. "Clientes. Caracterización": datos demográficos. Dos franjas. Primera con tarjetas de KPIs actuales en grande y debajo la distrubución de segmentos: Edad Media, Sexo, Membresías Familiares, Mismo CP (si su domicilio tiene el mismo CP que el centro), Permanencia. Segunda franja: dos evolutivos para ver tendencias: 1 evolutivo mensual dos últimos años de segmentos de edad y Edad Media. 2. evolutivo mensual dos últimos años de %Sexo femenino, %Mismo CP, Clientes con permanencia de más de 3 años tanto en valor absoluto como relativo
3. "Clientes.Visitas." caracterización de las visitas desde la óptica de cliente, no del centro. Dos franjas. Primer franja, tarjetas de KPIs actuales en grande y  y comparativos abajo más pequeño. Los KPIs son: Clientes Activos (los que han visitado el centro en el último mes), Visitas mes por activo (último mes), Duración de la Visita en minutos (último mes), Días desde la última visita. Debajo del KPI actual en mayúscula, está debajo más pequeño el valor del mes actual, del mes anterior, y de ese mes el año pasado. Segunda franja:  1 gráfico evolutivo mensual de los últimos dos años del %de clientes activos y su distribución por número de visitas. 2 gráfico evolutivo mensual últimos dos años de Distribución de Clientes actuales por segmentos de Edad-Sexo, en activos y distribución de número de visitas.
4. "Centro. Ocupación. Evolutivo Mensual". Visitas recibidas, duración, minutos de saturación del centro (concurrencia en el centro de más de 90 personas incluído el cliente). Primera franja: valores actuales de los KPIs y debajo los valores el mes anterior, hace dos meses, y mismo mes hace un año. Segunda franja. Evolutivo mensual de los últimos dos años de Visitas por mes, distribución de la ocupación mensual, y minutos de saturación por mes. Segunda Franja: Evolutivo mensual de Visitas de Clientes únicos, Duración promedio de visita por cliente, y Distribución de la Experiencia de Cliente. La Experiencia de Cliente se realiza en función del porcentaje de minutos de visita del cliente que no existió saturación.
5. "Centro. Ocupación. Evolutivo diario". Idem que el anterior pero diario y de los últimos 30 días. 
6. "Mapa de Calor de Ocupación día-hora". Concurrencia, Duración Visita, por Día de la Semana, Hora y Segmentadores de Sexo, Edad, Membresía, y CP. Está hecho por heatmap y no curva contínua porque se ve mejor los datos, permite colores, es más visual y con más datos. Permite conocer cuántas personas hay en cada hora y cuánto dura su visita, y emplear segmentadores si se quiere. Tiene además filas y columnas totales y promedios.
7. "Altas. Caracterización": datos demográficos. Dos franjas. Primera con tarjetas de KPIs: 1/ "Altas hoy" en grande, debajo en pequeño el mes acumulado, los últimos 30 días, el mes anterior, y el mismo período el año pasado. 2/de las altas en los útimos 30 días: Edad Media en grande (debajo en pequeño distribución por Sexo), Sexo Femenino% en grande (debajo más pequeño la distribución por sexos- franjas de edad), Membresías Familiares% (debajo distribución por tipo de Membresía), Mismo CP en grande, debajo más pequeño distribución por CP, ARPU en grande, debajo en pequeño ARPU acumulado del mes, del mes anterior, y del mismo mes el año pasado. Segunda franja: dos evolutivos para ver tendencias: 1 evolutivo mensual de las altas en los dos últimos años de segmentos de edad y Edad Media. 2. evolutivo mensual de altas dos últimos años de %Sexo femenino, %Mismo CP.
8. "Altas. Engagement.": visitas de las altas en los primeros 30 días. Primera Franja. Mismos KPIs que la página anterior. Segunda franja:  1 gráfico evolutivo mensual de los últimos dos años del %de altas activas y su distribución por número de visitas. 2 grafico evolutivo mensual últimos dos años de Distribución de Clientes actuales por segmentos de Edad-Sexo, en activos y distribución de número de visitas.
9. "Bajas. Caracterización": datos demográficos. Dos franjas. Primera con tarjetas de KPIs: 1/"Bajas hoy" en grande, debajo en pequeño el mes acumulado, los últimos 30 días, el mes anterior, y el mismo período el año pasado. 2/de las bajas en los útimos 30 días: Edad Media en grande (debajo en pequeño distribución por Sexo), Sexo Femenino% en grande (debajo más pequeño la distribución por sexos- franjas de edad), Membresías Familiares% (debajo distribución por tipo de Membresía), Mismo CP en grande, debajo más pequeño distribución por CP, ARPU en grande, debajo en pequeño ARPU acumulado del mes, del mes anterior, y del mismo mes el año pasado. Segunda franja: dos evolutivos para ver tendencias: 1 evolutivo mensual de las bajas en los dos últimos años de segmentos de edad y Edad Media. 2. evolutivo mensual de bajas en los dos últimos años de %Sexo femenino, %Mismo CP.
10. "Bajas. Engagement.". Bajas activas y visitas en el último mes. Primera Franja. Mismos KPIs que la página anterior. Segunda franja:  1 gráfico evolutivo mensual de los últimos dos años del %de bajas activas en el último mes y su distribución por número de visitas. 2 grafico evolutivo mensual últimos dos años de promdio días desde la última visita, y visitas promedio en el último mes.
11. "Altas. Cohortes". Agrupación de altas por cohortes mensuales, evolutivo mensual por antigüedad desde el mes 0 (mes del alta) hasta el mes 12, de datos de: % de Altas que permanecen como clientes, % de altas activas (con visitas ese mes), y número de visitas. Esta primera página es en forma matricial con totales y desglose por mes-año de cohorte para poder ver patrones y diferencias entre las cohortes.
12. "Altas. Cohortes. Resumen". Dos franjas. La primera es el evolutivo mensual por mes de antiguedad de Altas que se mantienen como clientes y de Altas que son activas (visitan el centro). Franja de abajo. Es el evolutivo mensual por mes de antigüedad de distribución de las altas por segmentos de experiencia (saturación del centro) y duración promedio de la visita en minutos. Como son gráficos agrupados por antigüedad, se han puesto arriba de la página segmentador por cohorte (mes-año), sexo ,segmento de edad, membresía, y Código Postal.
13. "Bajas. Cohortes". Agrupación de bajas por cohortes mensuales, evolutivo mensual por antigüedad inversa desde el mes 0 (mes de la baja) hasta el mes -12, de datos de: % de Bajas que permanecen como clientes, % de bajas activas (con visitas ese mes), y número de visitas. Esta primera página es en forma matricial con totales y desglose por mes-año de cohorte para poder ver patrones y diferencias entre las cohortes.
14. "Bajas. Cohortes. Resumen". Dos franjas. La primera es el evolutivo mensual por mes de antiguedad inversa de bajas que se mantienen como clientes y de bajas que son activas (visitan el centro). Franja de abajo. Es el evolutivo mensual por mes de antigüedad inversa de distribución de las bajas por segmentos de experiencia (saturación del centro) y duración promedio de la visita en minutos. Como son gráficos agrupados por antigüedad, se han puesto arriba de la página segmentador por cohorte (mes-año), sexo, segmento de edad, membresía, y Código Postal.